[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_rnaseq.ipynb) [![Open In nbviewer](https://img.shields.io/badge/View%20in-nbviewer-orange)](https://nbviewer.jupyter.org/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_rnaseq.ipynb)

# Bulk RNA-Seq

This tutorial is a brief guide for the implementation of BiT Age, a highly accurate bulk transcriptomic clock for C. elegans. Link to [paper](https://onlinelibrary.wiley.com/doi/full/10.1111/acel.13320).

We just need two packages for this tutorial.

In [1]:
import pandas as pd
import pyaging as pya

## Download and load example data

Let's download the C. elegans RNA-seq dataset from the BiT Age paper.

In [2]:
pya.data.download_example_data('GSE65765')

'pyaging_data/GSE65765_CPM.pkl'

In [3]:
df = pd.read_pickle('pyaging_data/GSE65765_CPM.pkl')

In [4]:
df.head()

,WBGene00197333,WBGene00198386,WBGene00015153,WBGene00002061,WBGene00255704,WBGene00235314,WBGene00001177,WBGene00169236,WBGene00219784,WBGene00015152,...,WBGene00010964,WBGene00014467,WBGene00014468,WBGene00014469,WBGene00014470,WBGene00010965,WBGene00014471,WBGene00010966,WBGene00010967,WBGene00014473
SRR1793993,0.0,0.0,3.780174,169.240815,1.907427,0.277444,59.320986,0.0,0.000000,1.283178,...,858.949156,0.0,0.000000,0.0,0.052021,234.526846,0.017340,54.483057,78.117815,0.000000
SRR1793991,0.0,0.0,0.510354,412.628597,0.061861,0.061861,22.239044,0.0,0.015465,0.201048,...,1049.982885,0.0,0.015465,0.0,0.015465,372.511713,0.000000,54.545971,59.618577,0.000000
SRR1793994,0.0,0.0,4.718708,274.733671,1.234644,0.118391,42.400721,0.0,0.000000,0.642691,...,664.255412,0.0,0.101478,0.0,0.000000,253.220421,0.033826,19.483698,86.492735,0.016913
SRR1793992,0.0,0.0,2.389905,351.612558,0.505892,0.069778,20.497358,0.0,0.017445,1.308342,...,1298.799849,0.0,0.034889,0.0,0.000000,472.206803,0.000000,89.508039,76.459508,0.000000


## Convert data to AnnData object

AnnData objects are highly flexible and are thus our preferred method of organizing data for age prediction.

In [5]:
adata = pya.preprocess.df_to_adata(df)

Note that the original DataFrame is stored in `X_original` under layers. is This is what the `adata` object looks like:

In [6]:
adata

AnnData object with n_obs × n_vars = 4 × 46755
    var: 'percent_na'
    layers: 'X_original', None (.X)

## Predict age

We can either predict one clock at once or all at the same time. Given we only have one clock of interest for this tutorial, let's go with one. The function is invariant to the capitalization of the clock name. 

In [7]:
pya.pred.predict_age(adata, 'BiTAge')

In [8]:
adata.obs.head()

,bitage
SRR1793993,182.353658
SRR1793991,27.337245
SRR1793994,241.629584
SRR1793992,32.178003


After age prediction, the clocks are added to `adata.obs`. Moreover, the percent of missing values for each clock and other metadata are included in `adata.uns`.

In [9]:
adata

AnnData object with n_obs × n_vars = 4 × 46755
    obs: 'bitage'
    var: 'percent_na'
    uns: 'bitage_percent_na', 'bitage_missing_features', 'bitage_supplied_features_mask', 'bitage_metadata'
    layers: 'X_original', None (.X)

## Get citation

The doi, citation, and some metadata are automatically added to the AnnData object under `adata.uns[CLOCKNAME_metadata]`.

In [10]:
adata.uns['bitage_metadata']

{'clock_name': 'bitage',
 'data_type': 'transcriptomics',
 'species': 'Caenorhabditis elegans',
 'year': 2021,
 'approved_by_author': '✅',
 'citation': 'Meyer, David H., and Björn Schumacher. "BiT age: A transcriptome-based aging clock near the theoretical limit of accuracy." Aging Cell 20 (2021): e13320.',
 'doi': 'https://doi.org/10.1111/acel.13320',
 'notes': 'Binarized whole-organism C. elegans RNA-seq clock that estimates temporally rescaled biological age; the released linear predictor sums coefficients for genes binarized on plus a 103.55-hour intercept.',
 'research_only': None,
 'tissue': ['whole organism'],
 'predicts': ['biological age'],
 'training_target': ['biological age'],
 'unit': ['hours'],
 'model_type': 'elastic net regression',
 'platform': ['RNA-seq'],
 'population': 'Caenorhabditis elegans',
 'journal': 'Aging Cell',
 'last_author': 'Björn Schumacher',
 'n_features': 576,
 'citations': 173,
 'citations_date': '2026-07-05',
 'version': 'v0.5.1',
 'preprocess': 'bi

## Cohort-relative clocks: tAge

`tage` and `tagemortality` come from Tyshkovskiy, Alexander, et al. "Universal transcriptomic hallmarks of mammalian ageing and mortality." *Nature* 654 (2026): 173-188. Link to [paper](https://doi.org/10.1038/s41586-026-10542-3). They are released under the MGB Open Access License 1.0 — **non-commercial academic research use only**, which is why the Clock Catalogue marks them "Research use only".

They cover mouse, rat, macaque, and human, so the C. elegans cohort above is not an input for them; the examples below assume your own bulk RNA-seq counts from one of those four species. Two things are worth understanding before running them.

**They read a cohort, not a sample.** Normalisation and centring are estimated across every sample in the call, so a sample's prediction depends on the samples predicted alongside it — the same sample scored in a different cohort gets a different number. Two samples is the hard minimum; below roughly ten the statistics are too noisy to read much into. `predict_age` runs this preprocessing itself on raw counts, so there is nothing to call from `pyaging.preprocess` first, and `adata.X` is left untouched.

**The prediction is a difference, not an age.** `tage` returns months of *mouse* age relative to the reference group: zero means "looks like the reference", negative means younger-looking. Mouse months are the unit whatever the species, because the model predicts a fraction of maximum lifespan and the mouse factor of 48 months is baked into the weights — rescale by your species' maximum lifespan over 48 (human 122.5 years, rat 50.4 months, macaque 39 years) to read it on its own timescale. `tagemortality` returns a base-10 log (log10) hazard ratio against the same reference.

### An illustrative cohort

The examples below need a cohort of one of those species to run on, so we build a small synthetic mouse one: eight samples, five hundred real mouse Ensembl gene IDs, counts drawn at random around a per-gene mean, and a `treatment` column splitting the animals into controls and treated. Only the gene identifiers are real — the counts carry no biology, so the numbers the clocks return below are there to show the mechanics and say nothing about ageing. Swap in your own count matrix and every cell that follows runs unchanged.

In [11]:
import numpy as np

# Real mouse Ensembl IDs, stored as their numeric suffixes to keep the cell short.
# Real identifiers matter here: pyaging maps them onto the mouse Entrez IDs the
# models are defined over, and invented ones would map to nothing.
GENE_SUFFIXES = """
00000000295,00000000561,00000000682,00000000827,00000000902,00000001053,00000001445,00000001555,00000001627,
00000001661,00000002108,00000002297,00000002409,00000002763,00000002778,00000003299,00000003420,00000003527,
00000003865,00000004056,00000004187,00000004221,00000004317,00000004364,00000004929,00000005125,00000005131,
00000005320,00000005580,00000005897,00000006310,00000006442,00000006527,00000006678,00000006731,00000007080,
00000007411,00000007892,00000008575,00000008892,00000009376,00000010044,00000010376,00000010406,00000011114,
00000012017,00000012117,00000012609,00000013787,00000014243,00000014294,00000014550,00000015023,00000015090,
00000015094,00000015112,00000016494,00000017057,00000017478,00000017781,00000018008,00000018378,00000018572,
00000018983,00000019659,00000019761,00000019768,00000019791,00000019843,00000020021,00000020022,00000020116,
00000020196,00000020198,00000020231,00000020340,00000020456,00000020459,00000020463,00000020527,00000020781,
00000020841,00000020900,00000021213,00000021281,00000021282,00000021339,00000021390,00000021481,00000021532,
00000021546,00000021738,00000021759,00000021785,00000021904,00000021998,00000022022,00000022236,00000022304,
00000022546,00000022621,00000022770,00000022787,00000022820,00000022868,00000022889,00000022898,00000022941,
00000022972,00000022978,00000023832,00000023882,00000023991,00000024174,00000024325,00000024381,00000024383,
00000024414,00000024431,00000024487,00000024667,00000024677,00000024694,00000024736,00000024737,00000024764,
00000024792,00000024841,00000024862,00000024875,00000024925,00000025154,00000025189,00000025290,00000025316,
00000025856,00000025915,00000025940,00000026031,00000026159,00000026175,00000026201,00000026219,00000026235,
00000026240,00000026249,00000026270,00000026319,00000026458,00000026649,00000026712,00000026827,00000027087,
00000027184,00000027185,00000027254,00000027463,00000027478,00000027668,00000027827,00000027835,00000027882,
00000028024,00000028033,00000028034,00000028339,00000028394,00000028412,00000028453,00000028496,00000028518,
00000028631,00000028639,00000028688,00000028759,00000028822,00000028948,00000029060,00000029068,00000029136,
00000029165,00000029175,00000029227,00000029260,00000029291,00000029405,00000029559,00000029686,00000029700,
00000029718,00000029725,00000029729,00000029765,00000029804,00000029862,00000030301,00000030342,00000030352,
00000030662,00000030703,00000030729,00000030752,00000031060,00000031157,00000031232,00000031400,00000031453,
00000031534,00000031563,00000031639,00000031673,00000031805,00000031864,00000031877,00000031928,00000031960,
00000031993,00000032035,00000032042,00000032068,00000032193,00000032244,00000032301,00000032324,00000032398,
00000032411,00000032513,00000032562,00000032582,00000032593,00000032606,00000032812,00000032846,00000032897,
00000032932,00000032942,00000033021,00000033065,00000033088,00000033633,00000033658,00000033715,00000033909,
00000033948,00000033965,00000034120,00000034463,00000034601,00000034648,00000034744,00000034762,00000034875,
00000034947,00000035086,00000035212,00000035235,00000035296,00000035407,00000035673,00000035697,00000035735,
00000035770,00000035845,00000035878,00000035885,00000036053,00000036241,00000036285,00000036528,00000036615,
00000036639,00000036641,00000036731,00000036943,00000036948,00000036975,00000036989,00000037104,00000037411,
00000037773,00000038127,00000038201,00000038417,00000038485,00000038489,00000038495,00000038506,00000038508,
00000038623,00000038685,00000038717,00000038721,00000038843,00000038855,00000038895,00000038930,00000039202,
00000039246,00000039253,00000039607,00000039639,00000039678,00000039713,00000039982,00000040044,00000040105,
00000040177,00000040263,00000040265,00000040283,00000040339,00000040363,00000040669,00000040687,00000040761,
00000040855,00000040985,00000041012,00000041164,00000041193,00000041248,00000041654,00000041939,00000042046,
00000042203,00000042284,00000042293,00000042303,00000042505,00000042712,00000042743,00000043085,00000043219,
00000043411,00000043518,00000044066,00000044254,00000044258,00000044847,00000044864,00000045136,00000045252,
00000045282,00000045319,00000045374,00000045503,00000045969,00000045980,00000046345,00000046402,00000046408,
00000046667,00000047146,00000047539,00000047694,00000048065,00000048310,00000048424,00000048478,00000048572,
00000048661,00000049288,00000049303,00000049422,00000049760,00000049775,00000049878,00000050022,00000050097,
00000050144,00000050608,00000051236,00000051256,00000051439,00000051615,00000052188,00000052384,00000052688,
00000052837,00000052957,00000053390,00000053414,00000053646,00000054021,00000054659,00000054942,00000055003,
00000055013,00000055024,00000055320,00000055435,00000055493,00000056076,00000056234,00000056429,00000056598,
00000056917,00000057110,00000058503,00000058706,00000058709,00000059355,00000060216,00000060438,00000060739,
00000060913,00000060935,00000061028,00000061477,00000061479,00000061650,00000062300,00000062515,00000062694,
00000063065,00000063358,00000063362,00000063659,00000063663,00000063849,00000066512,00000066705,00000067144,
00000067219,00000068551,00000068923,00000069456,00000069769,00000070283,00000070284,00000070420,00000070427,
00000070520,00000070583,00000070934,00000071456,00000071713,00000072235,00000072849,00000073940,00000074030,
00000075025,00000075520,00000075590,00000075702,00000076435,00000078486,00000078648,00000078779,00000079037,
00000079105,00000079111,00000079293,00000079610,00000083992,00000084106,00000084111,00000085541,00000085645,
00000085665,00000085779,00000086003,00000086567,00000086746,00000086859,00000086868,00000087177,00000089832,
00000089889,00000090213,00000091144,00000091387,00000091811,00000093930,00000094686,00000095253,00000095687,
00000096606,00000097039,00000097174,00000097321,00000101304,00000102976,00000103088,00000103160,00000104459,
00000108322,00000110540,00000111186,00000112041,00000113178,00000116996,00000118559,00000120047,00000120146,
00000120183,00000120456,00000120992,00000121053,00002076161
"""

genes = ["ENSMUSG" + suffix for suffix in "".join(GENE_SUFFIXES.split()).split(",")]
samples = [f"sample_{i:02d}" for i in range(1, 9)]

rng = np.random.default_rng(0)
gene_means = rng.lognormal(mean=4.0, sigma=1.2, size=len(genes))
counts = pd.DataFrame(
    rng.poisson(gene_means * rng.lognormal(0.0, 0.3, size=(len(samples), len(genes)))),
    index=samples,
    columns=genes,
)
counts["treatment"] = ["control"] * 4 + ["treated"] * 4
counts.iloc[:5, :5]

,ENSMUSG00000000295,ENSMUSG00000000561,ENSMUSG00000000682,ENSMUSG00000000827,ENSMUSG00000000902
sample_01,97,64,72,44,34
sample_02,80,60,125,78,13
sample_03,96,35,81,83,25
sample_04,69,43,89,37,12
sample_05,45,24,81,48,44


### A default run

The input is a raw count matrix, samples by genes, indexed by gene identifiers of your cohort's own species — symbols, Ensembl, or Entrez IDs all work. The models are defined over mouse Entrez IDs and pyaging maps your identifiers onto them, so a human cohort is labelled with human genes, not translated by hand; anything the models expect but your data does not measure falls back to its training median. That fallback is why the run below reports most of its features as missing: five hundred genes is a fraction of the roughly ten thousand the models read, and a real cohort quotes a far smaller number. Sample annotations can ride along in the same frame: `metadata_cols` moves those columns into `.obs` instead of reading them as genes, and the `treatment` column is what the reference-group example further down selects on.

In [12]:
mouse_adata = pya.preprocess.df_to_adata(counts, metadata_cols=['treatment'])
pya.pred.predict_age(mouse_adata, ['tage', 'tagemortality'])
mouse_adata.obs[['tage', 'tagemortality']]

src/pyaging/preprocess/_tage.py:209: UserWarning: no species indicator column found; defaulting to mouse (add a column named mouse/rat/macaque/human, set to 1 for every sample, to say otherwise)
  _warn(


,tage,tagemortality
sample_01,0.717187,0.101780
sample_02,1.321674,0.031420
sample_03,0.853895,0.079172
sample_04,1.812788,0.174575
sample_05,-0.357311,0.014223
sample_06,0.384510,0.107217
sample_07,0.230006,0.047854
sample_08,0.983944,0.052607


With no reference group named, the cohort centres on itself, so each prediction says how that sample's expression compares with the cohort as a whole. Note that it is the expression that is centred, not the predictions: nothing pins the cohort's mean prediction to zero, and the eight above average about +0.74 mouse-months. That shared offset carries no meaning — what does is the spacing, how far each sample sits from the others.

### Naming the species

The run above never said which species the cohort is. With no species column pyaging assumes mouse and logs a warning — raised both on the predict display and as a Python `UserWarning`, so `verbose=False` does not hide it. Name the species explicitly in anything scripted rather than relying on the warning.

The idiom is a column set to `1` for every sample, the same one the mammalian methylation clocks use for covariates such as `female`. Valid names are `mouse`, `rat`, `macaque`, and `human`, matched case-insensitively. Our cohort is mouse, so saying so changes nothing but the warning:

In [13]:
mouse_counts = counts.copy()
mouse_counts['mouse'] = 1
mouse_adata = pya.preprocess.df_to_adata(mouse_counts, metadata_cols=['treatment'])
pya.pred.predict_age(mouse_adata, ['tage', 'tagemortality'])
mouse_adata.obs[['tage', 'tagemortality']]

,tage,tagemortality
sample_01,0.717187,0.101780
sample_02,1.321674,0.031420
sample_03,0.853895,0.079172
sample_04,1.812788,0.174575
sample_05,-0.357311,0.014223
sample_06,0.384510,0.107217
sample_07,0.230006,0.047854
sample_08,0.983944,0.052607


For a human cohort you would instead label it with human gene identifiers and set the `human` column:

```python
human_counts['human'] = 1
human_adata = pya.preprocess.df_to_adata(human_counts, metadata_cols=['treatment'])
pya.pred.predict_age(human_adata, ['tage', 'tagemortality'])
```

Exactly one indicator may be set, and the column is dropped before the gene pipeline rather than read as a gene — which is why it goes in the count frame rather than in `metadata_cols`. Setting two, or letting one vary between samples, is an error rather than a guess.

### Choosing the reference group

Centring on the whole cohort answers "which of these samples look older than the others". Usually the more useful question is "how do the treated animals look next to the controls", which means centring on a subset. Mark it with a truthy `adata.obs['tage_reference_group']`, boolean or `0`/`1`.

In [14]:
mouse_adata.obs['tage_reference_group'] = mouse_adata.obs['treatment'] == 'control'
pya.pred.predict_age(mouse_adata, ['tage', 'tagemortality'])
mouse_adata.obs[['treatment', 'tage', 'tagemortality']]

,treatment,tage,tagemortality
sample_01,control,0.491061,0.095391
sample_02,control,1.095548,0.025031
sample_03,control,0.627769,0.072783
sample_04,control,1.586663,0.168186
sample_05,treated,-0.583436,0.007834
sample_06,treated,0.158384,0.100828
sample_07,treated,0.003881,0.041465
sample_08,treated,0.757818,0.046218


Changing the reference group shifts every prediction by the same amount and leaves the spread between samples alone — what moves is the baseline the deviations are measured from, now the controls rather than the cohort mean. So it is differences that carry the meaning: a treated animal scoring 3.2 mouse-months below the control average is 3.2 months of mouse age younger-looking than those controls. A column that selects no samples raises rather than quietly falling back to the whole cohort.

What the preprocessing actually did — the species it used, how many genes mapped, how large the reference group was — is recorded in `adata.uns['tage_preparation']`, and the usual `adata.uns['tage_metadata']` carries the citation and licence.

In [15]:
mouse_adata.uns['tage_preparation']

{'species': 'mouse',
 'n_input_genes': 500,
 'n_filtered_genes': 482,
 'n_mapped_genes': 456,
 'n_reference_samples': 4,
 'reference_group': ['sample_01', 'sample_02', 'sample_03', 'sample_04']}